In [2]:
import pandas as pd

df = pd.read_csv(
    "main_df_train.csv"

)

print(df.shape)

/tmp/ipykernel_16943/3119900830.py:3: DtypeWarning: Columns (74,115) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


(808387, 301)


In [3]:
df = df[df["MONTH"].isin([-1, 12])]

print("After filter:", df.shape)

After filter: (109690, 301)


In [4]:
y = df["HighCostLabel"]
y.shape

(109690,)

In [5]:
df = df.drop(columns=["HighCostLabel", "NEXT_YEAR_COST"], errors="ignore")

In [6]:
member_ids = df["Member_Key"]
df = df.drop(columns=["Member_Key"])

In [7]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in num_cols:
    df[col] = df[col].clip(lower=0)

In [8]:
missing_ratio = df.isnull().sum() / len(df)
missing_ratio = missing_ratio.sort_values(ascending=False)
print(missing_ratio.head(20))

AttributionStatus                1.000000
PersiviaMemberHccScore           1.000000
Ishighrisk                       0.622937
PersiviaCost                     0.587100
ActualMonthNumber                0.414377
ActualYear                       0.414377
AWVLastDate                      0.269906
AWVCode                          0.269906
AWVProviderNetwork               0.269906
RxBrandPrescription              0.096481
OtherOutpatientInNetwork_Cost    0.096481
ProfessionalER_Visit             0.096481
OfficeCost                       0.096481
CAH2OutpatientInNetwork_Cost     0.096481
AmbulatorySurgeryCost            0.096481
AmbulatorySurgeryVisit           0.096481
UrgentCareCost                   0.096481
RHCOutpatientInNetwork_Cost      0.096481
UrgentCareVisit                  0.096481
OONPreferredCost                 0.096481
dtype: float64


In [9]:
high_missing_cols = missing_ratio[missing_ratio > 0.8].index
df = df.drop(columns=high_missing_cols)

print("Dropped:", len(high_missing_cols))

Dropped: 2


In [10]:
medium_cols = missing_ratio[(missing_ratio > 0.4) & (missing_ratio <= 0.8)].index

for col in medium_cols:
    df[col + "_missing"] = df[col].isnull().astype(int)

    if df[col].dtype == "object":
        df[col] = df[col].fillna("Unknown")
    else:
        df[col] = df[col].fillna(df[col].median())

/tmp/ipykernel_16943/533546685.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isnull().astype(int)
/tmp/ipykernel_16943/533546685.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isnull().astype(int)
/tmp/ipykernel_16943/533546685.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a d

In [11]:
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == "object":
            df[col] = df[col].fillna("Unknown")
        else:
            df[col] = df[col].fillna(df[col].median())

In [12]:
from sklearn.preprocessing import LabelEncoder

cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

In [13]:
X = df.copy()

print("Final X:", X.shape)
print("Final y:", y.shape)

Final X: (109690, 300)
Final y: (109690,)


In [14]:
print("Missing values:", X.isnull().sum().sum())
print("Target distribution:\n", y.value_counts())

Missing values: 0
Target distribution:
 HighCostLabel
0.0    97429
1.0    12261
Name: count, dtype: int64


In [15]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape, X_val.shape)

(87752, 300) (21938, 300)


In [16]:
!pip install lightgbm

In [17]:
from lightgbm import LGBMClassifier

model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 9809, number of negative: 77943
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.742381 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25613
[LightGBM] [Info] Number of data points in the train set: 87752, number of used features: 249
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


LGBMClassifier(class_weight='balanced', learning_rate=0.05, n_estimators=300,
               random_state=42)

In [18]:
y_pred = model.predict(X_val)
y_prob = model.predict_proba(X_val)[:, 1]

In [19]:
from sklearn.metrics import f1_score, classification_report, roc_auc_score

print("F1 (default 0.5):", f1_score(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_prob))

print(classification_report(y_val, y_pred))

F1 (default 0.5): 0.3927629099133057
ROC-AUC: 0.790876379394149
              precision    recall  f1-score   support

         0.0       0.95      0.80      0.87     19486
         1.0       0.28      0.64      0.39      2452

    accuracy                           0.78     21938
   macro avg       0.61      0.72      0.63     21938
weighted avg       0.87      0.78      0.81     21938



In [20]:
import numpy as np

best_f1 = 0
best_thresh = 0

for t in np.arange(0.05, 0.7, 0.02):
    preds = (y_prob >= t).astype(int)
    f1 = f1_score(y_val, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print("Best Threshold:", best_thresh)
print("Best F1:", best_f1)

Best Threshold: 0.6500000000000001
Best F1: 0.4200757575757576


In [21]:
best_thresh = 0.65

In [22]:
final_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    class_weight="balanced",
    random_state=42
)

final_model.fit(X, y)

[LightGBM] [Info] Number of positive: 12261, number of negative: 97429
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.224675 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 26214
[LightGBM] [Info] Number of data points in the train set: 109690, number of used features: 249
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


LGBMClassifier(class_weight='balanced', learning_rate=0.05, n_estimators=300,
               random_state=42)

In [29]:
df_test = pd.read_csv("main_df_test.csv")

In [30]:
df_test = df_test[df_test["MONTH"].isin([-1, 12])]

df_test = df_test.drop(columns=["NEXT_YEAR_COST"], errors="ignore")

# ✅ SAVE FIRST
test_member_ids = df_test["Member_Key"]

# ✅ DROP AFTER
df_test = df_test.drop(columns=["Member_Key"], errors="ignore")

# CONTINUE
num_cols = df_test.select_dtypes(include=['int64', 'float64']).columns
for col in num_cols:
    df_test[col] = df_test[col].clip(lower=0)

for col in df_test.columns:
    if df_test[col].dtype == "object":
        df_test[col] = df_test[col].fillna("Unknown")
    else:
        df_test[col] = df_test[col].fillna(df_test[col].median())

from sklearn.preprocessing import LabelEncoder
cat_cols = df_test.select_dtypes(include=['object']).columns

for col in cat_cols:
    df_test[col] = LabelEncoder().fit_transform(df_test[col].astype(str))

# ADD MISSING COLUMNS
for col in X.columns:
    if col not in df_test.columns:
        df_test[col] = 0

# ALIGN
df_test = df_test[X.columns]

/tmp/ipykernel_16943/3089206138.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_test[col] = 0
/tmp/ipykernel_16943/3089206138.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_test[col] = 0
/tmp/ipykernel_16943/3089206138.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_test[col]

In [31]:
test_prob = final_model.predict_proba(df_test)[:, 1]
test_pred = (test_prob >= best_thresh).astype(int)

In [32]:
submission = pd.DataFrame({
    "Member_Key": test_member_ids,
    "HighCostLabel": test_pred
})

submission.to_csv("submission.csv", index=False)

print(submission.shape)

(12387, 2)
